# IdeaWeaver SLM Builder

Configure a from-scratch, Gemma-4-Nano-style small language model — interleaved local/global attention, grouped-query attention, QK-norm, partial RoPE, and cross-layer KV-cache sharing — and watch it actually train on TinyStories, with a real, live loss curve.

**To run this, press "Runtime" → "Run all" on a Tesla T4 Google Colab instance!** A GPU isn't strictly required (the backend falls back to CPU), but training is far too slow on CPU to be useful — use a T4 or better: *Runtime → Change runtime type → T4 GPU*.

[GitHub](https://github.com/ideaweaver-ai/ideaweaver-slm-builder) · [IdeaWeaver AI Labs](https://www.ideaweaver.ai) · [Building Small Language Models from Scratch (course)](https://www.ideaweaver.ai/courses)

> This notebook clones the repo, installs the Node.js frontend and Python training backend, starts both, and displays the app below as an embedded iframe — the same `serve_kernel_port_as_iframe` pattern Unsloth Studio uses. **"Start Training" is real**: it builds the exact model you configure and trains it on TinyStories, streaming real loss back into the chart. The first run downloads and tokenizes TinyStories (~10–20 minutes); every run after that reuses the cached tokenizer and data.

### Setup: clone the repo

In [ ]:
!git clone --depth 1 --branch main https://github.com/ideaweaver-ai/ideaweaver-slm-builder.git
%cd ideaweaver-slm-builder

### Install Node.js 20 and project dependencies

Colab's base image doesn't ship a recent-enough Node.js for Next.js 16, so this installs Node 20 from NodeSource first.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs > /dev/null 2>&1
!node -v && npm -v
!npm install --no-audit --no-fund

### Install the Python training backend's dependencies

Colab already has `torch` with CUDA support preinstalled — this deliberately does **not** touch `torch` or `numpy`, only adds what's missing (FastAPI, Uvicorn, SentencePiece, Hugging Face `datasets`), so the preinstalled CUDA-enabled PyTorch build is never reinstalled or downgraded.

In [ ]:
!pip install --quiet fastapi uvicorn sentencepiece datasets tqdm

### Start the training backend

Starts `train_service.py` (FastAPI) in the background on port 8001 and waits for its `/health` check to pass. The frontend (started in the next cell) talks to this over `http://127.0.0.1:8001` via its own API routes — the browser never calls this port directly.

In [ ]:
import subprocess, threading, time, urllib.request

BACKEND_PORT = 8001

backend_proc = subprocess.Popen(
    ["uvicorn", "train_service:app", "--host", "127.0.0.1", "--port", str(BACKEND_PORT)],
    cwd="backend",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def _pump_backend():
    for line in backend_proc.stdout:
        print(line, end="")

threading.Thread(target=_pump_backend, daemon=True).start()

def _backend_healthy():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{BACKEND_PORT}/health", timeout=1) as r:
            return r.status == 200
    except Exception:
        return False

for _ in range(60):
    if _backend_healthy():
        print("\nTraining backend is up on port", BACKEND_PORT)
        break
    time.sleep(1)
else:
    raise RuntimeError("Training backend didn't start in time — check the log output above.")

### Start the frontend

Starts the Next.js dev server in the background, waits for it to be ready, then displays it in an embedded iframe below — the same `serve_kernel_port_as_iframe` trick Unsloth Studio uses to run a local web app inside a Colab cell. It talks to the backend started above over its own `/api/train/*` routes.

In [ ]:
import subprocess, threading, time

PORT = 3000

proc = subprocess.Popen(
    ["npm", "run", "dev", "--", "-p", str(PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

ready = threading.Event()

def _pump():
    for line in proc.stdout:
        print(line, end="")
        if "Ready in" in line:
            ready.set()

threading.Thread(target=_pump, daemon=True).start()

if not ready.wait(timeout=60):
    raise RuntimeError("Server didn't start in time — check the log output above.")

time.sleep(1)  # small buffer so the first request doesn't race the server

from google.colab import output
output.serve_kernel_port_as_iframe(PORT, height=1200, width="100%")

---

Click **▶ Start Training** in the app above — the first run will spend several minutes downloading and tokenizing TinyStories before the loss curve starts moving (watch the status line under the chart). You can **Stop** anytime and download the checkpoint it's saved so far.

Questions or bugs? Open an issue on [GitHub](https://github.com/ideaweaver-ai/ideaweaver-slm-builder/issues). Want to understand every one of these variables — attention internals, RoPE, KV caching, and the training loop — not just tune them? Check out [Building Small Language Models from Scratch](https://www.ideaweaver.ai/courses).